# Wan Studio Colab Web UI

Run this notebook when you want Colab to host Wan Studio and generate real Wan videos.

1. Select a GPU runtime in Colab. Start with the 8GB build; 16GB and 24GB builds are available from the Web UI.
2. Clone the GitHub repo and install Wan Studio.
3. Mount Google Drive and download or reuse the Wan2.2 I2V A14B base model, LightX2V low-VRAM files, and optional LoRA set.
4. Install the LightX2V low-VRAM runner.
5. Start the Web UI in LightX2V runner mode and keep the final cell running.


## 1. Clone and install Wan Studio

Run this first. It clones the public GitHub repo into the temporary Colab runtime and installs the app.


In [ ]:
REPO_URL = 'https://github.com/jjj06960-hash/wan-studio.git'
APP_DIR = '/content/wan-studio'

!rm -rf {APP_DIR}
!git clone {REPO_URL} {APP_DIR}
%cd {APP_DIR}
!python install.py --accelerator cuda --system


## 2. Mount Drive and download the real base model, low-VRAM files, and LoRA set once

The LoRA/adapters repo is not enough for generation by itself. This cell downloads the default A14B I2V base, LightX2V 4-step FP8 files, and the default LoRA set to Drive.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

A14B_I2V_REPO_ID = 'Wan-AI/Wan2.2-I2V-A14B'
A14B_I2V_MODEL_DIR = Path('/content/drive/MyDrive/WanStudio/models/Wan2.2-I2V-A14B')
LIGHTX2V_REPO_ID = 'lightx2v/Wan2.2-Distill-Models'
LIGHTX2V_MODEL_DIR = Path('/content/drive/MyDrive/WanStudio/models/Wan2.2-LightX2V')
LORA_REPO_ID = 'lkzd7/WAN2.2_LoraSet_NSFW'
LORA_MODEL_DIR = Path('/content/drive/MyDrive/WanStudio/models/WAN2.2_LoraSet_NSFW')
WEIGHT_SUFFIXES = {'.safetensors', '.bin', '.pt', '.pth', '.ckpt', '.gguf'}

!pip install -U "huggingface_hub[cli]"

def has_weights(model_dir):
    return model_dir.exists() and any(
        path.suffix in WEIGHT_SUFFIXES for path in model_dir.rglob('*') if path.is_file()
    )

if has_weights(A14B_I2V_MODEL_DIR):
    print('A14B I2V base model already exists in Drive:', A14B_I2V_MODEL_DIR)
else:
    !hf download {A14B_I2V_REPO_ID} --local-dir {A14B_I2V_MODEL_DIR}

if has_weights(LIGHTX2V_MODEL_DIR):
    print('LightX2V low-VRAM files already exist in Drive:', LIGHTX2V_MODEL_DIR)
else:
    !hf download {LIGHTX2V_REPO_ID} --local-dir {LIGHTX2V_MODEL_DIR} --include "wan2.2_i2v_A14b_*_noise_scaled_fp8_e4m3_lightx2v_4step.safetensors"

if has_weights(LORA_MODEL_DIR):
    print('LoRA set already exists in Drive:', LORA_MODEL_DIR)
else:
    !hf download {LORA_REPO_ID} --local-dir {LORA_MODEL_DIR}

print('Use this base model folder in Wan Studio:', A14B_I2V_MODEL_DIR)
print('Low-VRAM files folder:', LIGHTX2V_MODEL_DIR)
print('Optional LoRA folder:', LORA_MODEL_DIR)


## 3. Install the LightX2V low-VRAM runner

Wan Studio uses LightX2V for the 8GB, 16GB, and 24GB builds. The official Wan repository is still cloned for model code compatibility, but the Web UI launches with `--runner lightx2v`.


In [ ]:
WAN_REPO_DIR = '/content/Wan2.2'
LIGHTX2V_REPO_DIR = '/content/LightX2V'

!rm -rf {WAN_REPO_DIR}
!git clone https://github.com/Wan-Video/Wan2.2.git {WAN_REPO_DIR}
%cd {WAN_REPO_DIR}
!pip install -r requirements.txt

!rm -rf {LIGHTX2V_REPO_DIR}
!git clone https://github.com/ModelTC/LightX2V.git {LIGHTX2V_REPO_DIR}
%cd {LIGHTX2V_REPO_DIR}
!pip install -v .


## 4. Launch the Web UI in real generation mode

Keep this cell running. Colab will show the Web UI iframe and print an `Open Wan Studio Web UI:` proxy link. Use that proxy link, not a `0.0.0.0` or `127.0.0.1` link.

In the Web UI, connect `/content/drive/MyDrive/WanStudio/models/Wan2.2-I2V-A14B`, choose `8GB`, `16GB`, or `24GB`, then add an image reference and prompt. To attach an adapter, paste `/content/drive/MyDrive/WanStudio/models/WAN2.2_LoraSet_NSFW` into `LoRA adapter folder or file`, scan, and pick a compatible embedded preset. LOW/HIGH pairs are grouped automatically.


In [ ]:
%cd /content/wan-studio
!python wan_studio.py run --host 127.0.0.1 --port 7860 --share --runner lightx2v


## Notes

- `lkzd7/WAN2.2_LoraSet_NSFW` is a LoRA/adapters set, not a standalone base model. The Web UI can attach selected `.safetensors` files as a LoRA layer.
- The official A14B runner is heavy. The beginner path is the 8GB/16GB/24GB LightX2V build selector.
- Generated files are written to `/content/wan-studio/outputs` and linked in the Jobs panel.
